# 📡 Step 1: Data Collection

This notebook handles:
1. Fetching player statistics from the **NBA Official API** (2025-26 season)
2. Scraping salary data from **Basketball Reference**
3. Cleaning and merging both datasets into a single analysis-ready dataframe

**Output:** `data/nba_stats_2025_26.csv` and `data/nba_salaries_2026_27.csv`

## 1.1 Fetch Player Statistics from NBA API

In [1]:
import pandas as pd
from nba_api.stats.endpoints import LeagueDashPlayerStats

print("Fetching data from NBA servers (Please wait)...")

# headers and timeout parts are completely removed.
stats = LeagueDashPlayerStats(
    season='2025-26',
    per_mode_detailed='PerGame'
)

# Convert JSON to dataframe
df_raw = stats.get_data_frames()[0]

print(f"Data successfully fetched! Total Players: {df_raw.shape[0]}")

# We use display() to view the table properly (as HTML) in Jupyter


Fetching data from NBA servers (Please wait)...
Data successfully fetched! Total Players: 582


## 1.2 Filter and Select Useful Features

In [2]:
import numpy as np

# 1. Filter those who played at least 15 games
df_filtered = df_raw[df_raw["GP"] > 15].copy()

# 2. Automatically get ALL numeric columns in the dataframe (Including all stats and ALL RANKs!)
all_numeric = df_filtered.select_dtypes(include=[np.number]).columns.tolist()

# 3. WE ARE THROWING AWAY ONLY AND ONLY THE REAL TRASH:
# - TEAM_ID: 1.6 Billion fake barcode number
# - WNBA fantasy points
real_trash = ['TEAM_ID']
clean_numeric = [col for col in all_numeric if col not in real_trash]

# 4. Add Name and Team information
must_have = ['PLAYER_ID', 'PLAYER_NAME', 'TEAM_ABBREVIATION']
selected_columns = must_have + [col for col in clean_numeric if col not in must_have]

# 5. Create and save the dataframe
df_stats = df_filtered[selected_columns].copy()
df_stats.to_csv("../data/nba_stats_2025_26.csv", index=False)

print(f"\u2705 Filtered Player Count: {df_stats.shape[0]}")
print(f"\u2705 Total Feature Count  : {len(selected_columns)} (HUGE roster including ranks!)")

✅ Filtered Player Count: 464
✅ Total Feature Count  : 67 (HUGE roster including ranks!)


## 1.3 Scrape Salary Data from Basketball Reference

In [3]:
print("now we are getting the current player payments")
url='https://www.basketball-reference.com/contracts/players.html'
dfs=pd.read_html(url)
df_salaries_raw=dfs[0]
df_salaries_raw.columns = df_salaries_raw.columns.droplevel(0)

now we are getting the current player payments


In [4]:
df_salaries = df_salaries_raw[['Player', '2026-27']].copy()
df_salaries.columns = ['PLAYER_NAME', 'SALARY']
df_salaries = df_salaries.drop_duplicates(subset=['PLAYER_NAME'], keep='first')
df_salaries = df_salaries.dropna(subset=['PLAYER_NAME', 'SALARY'])
df_salaries = df_salaries[df_salaries['PLAYER_NAME'] != 'Player']

## 1.4 Clean Salary Data and Remove NaNs

In [5]:
df_salaries['SALARY'] = pd.to_numeric(
    df_salaries['SALARY'].str.replace('$', '').str.replace(',', ''), 
    errors='coerce'
)
print(df_salaries.isna().sum())
print(df_salaries.isna().sum())
print(df_stats["PLAYER_NAME"].nunique())
print(df_stats["PLAYER_NAME"].size)
print(df_salaries["PLAYER_NAME"].nunique())
print(df_salaries["PLAYER_NAME"].size)
print(df_salaries["PLAYER_NAME"].value_counts())
print(df_salaries)

PLAYER_NAME    0
SALARY         0
dtype: int64
PLAYER_NAME    0
SALARY         0
dtype: int64
464
464
477
477
PLAYER_NAME
Stephen Curry            1
Nikola Jokić             1
Jayson Tatum             1
Anthony Davis            1
Giannis Antetokounmpo    1
                        ..
Vasilije Micić           1
Mamadi Diakite           1
Ricky Rubio              1
Didi Louzada             1
Taj Gibson               1
Name: count, Length: 477, dtype: int64
               PLAYER_NAME    SALARY
0            Stephen Curry  62587158
1             Nikola Jokić  59033114
2             Jayson Tatum  58456566
3            Anthony Davis  58456566
4    Giannis Antetokounmpo  58456566
..                     ...       ...
525         Vasilije Micić    666667
528         Mamadi Diakite    464050
529            Ricky Rubio    424672
530           Didi Louzada    268032
531             Taj Gibson    148828

[477 rows x 2 columns]


In [6]:
df_salaries.to_csv("../data/nba_salaries_2026_27.csv", index=False)
print("the salary is saved")

the salary is saved


## 1.5 Merge Stats and Salary Data

In [7]:
import unicodedata

def clean_name_simple(name):
    if pd.isna(name):
        return name
    name = unicodedata.normalize('NFKD', name).encode('ASCII', 'ignore').decode('utf-8')
    name=name.replace('.','')
    return name.upper().strip()

df_salaries["PLAYER_NAME"]=df_salaries["PLAYER_NAME"].apply(clean_name_simple)
df_stats["PLAYER_NAME"]=df_stats["PLAYER_NAME"].apply(clean_name_simple)
df_inner=pd.merge(df_stats,df_salaries,on="PLAYER_NAME",how="inner")
print(df_inner)
print(df_inner.shape[0])

     PLAYER_ID         PLAYER_NAME TEAM_ABBREVIATION   AGE  GP   W   L  W_PCT  \
0      1631260            AJ GREEN               MIL  26.0  78  31  47  0.397   
1      1642358          AJ JOHNSON               DAL  21.0  48   9  39  0.188   
2       203932        AARON GORDON               DEN  30.0  36  27   9  0.750   
3      1630174       AARON NESMITH               IND  26.0  45  10  35  0.222   
4      1630598       AARON WIGGINS               OKC  27.0  65  50  15  0.769   
..         ...                 ...               ...   ...  ..  ..  ..    ...   
375    1642258  ZACCHARIE RISACHER               ATL  21.0  67  39  28  0.582   
376     203897         ZACH LAVINE               SAC  31.0  39   9  30  0.231   
377    1630192          ZEKE NNAJI               DEN  25.0  52  38  14  0.731   
378    1630533     ZIAIRE WILLIAMS               BKN  24.0  56  13  43  0.232   
379    1629627     ZION WILLIAMSON               NOP  25.0  62  22  40  0.355   

      MIN  FGM  ...  PFD_RA

---
**Next Step:** Open `02_eda_feature_engineering.ipynb` for exploratory data analysis and feature selection.